In [1]:
import numpy as np
import pandas as pd
import os
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None

In [ ]:
# Prima esplora lo schema reale di DiaObject
query_schema = """
SELECT column_name, datatype, description
FROM TAP_SCHEMA.columns
WHERE table_name = 'dp1.DiaObject'
ORDER BY column_name
"""

job = service.submit_job(query_schema)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
schema = job.fetch_result().to_table().to_pandas()
#print(schema.to_string())
mask = (
    schema['column_name'].str.contains('Flux', case=False) |
    schema['column_name'].str.contains('Sigma', case=False) |
    schema['column_name'].str.contains('nDia', case=False)
)
print(schema[mask][['column_name', 'datatype', 'description']].to_string())

In [ ]:
import pandas as pd
import os
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None

# ─────────────────────────────────────────────
# Parametri di selezione
# ─────────────────────────────────────────────
N_SOURCES_MIN  = 20
FLUX_SIGMA_MIN = 500.0
STETSON_MIN    = 0.2

query_objects = f"""
SELECT
    diaObjectId,
    ra,
    dec
FROM dp1.DiaObject
WHERE nDiaSources >= {N_SOURCES_MIN}
  AND (
       (g_psfFluxSigma > {FLUX_SIGMA_MIN} AND g_psfFluxStetsonJ > {STETSON_MIN})
    OR (r_psfFluxSigma > {FLUX_SIGMA_MIN} AND r_psfFluxStetsonJ > {STETSON_MIN})
    OR (i_psfFluxSigma > {FLUX_SIGMA_MIN} AND i_psfFluxStetsonJ > {STETSON_MIN})
  )
"""

print("Cerco oggetti variabili...")
job = service.submit_job(query_objects)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase:', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

dia_objects = job.fetch_result().to_table().to_pandas()
print(f"Trovati {len(dia_objects)} oggetti variabili candidati")

# Salva CSV
os.makedirs("./light_curves", exist_ok=True)
out_path = "./variable_objects_catalog_all.csv"
dia_objects.to_csv(out_path, index=False)
print(f"Salvato: {out_path}")
print(dia_objects.head(10).to_string())

In [ ]:
# ─────────────────────────────────────────────
# STEP 1 — Trova stelle variabili in DiaObject
# ─────────────────────────────────────────────
N_SOURCES_MIN  = 30      # minimo numero di osservazioni
FLUX_SIGMA_MIN = 500.0   # variabilità minima in nJy
MAX_OBJECTS    = 50000

# Usiamo anche StetsonJ > 0.5 come indicatore di variabilità reale
# (robusto contro outlier, buono per RR Lyrae / Cefeidi / EB)
STETSON_MIN = 0.5

query_objects = f"""
SELECT TOP {MAX_OBJECTS}
    diaObjectId,
    ra, dec,
    nDiaSources,
    g_psfFluxSigma, r_psfFluxSigma, i_psfFluxSigma,
    g_psfFluxMean,  r_psfFluxMean,  i_psfFluxMean,
    g_psfFluxStetsonJ, r_psfFluxStetsonJ, i_psfFluxStetsonJ,
    g_psfFluxChi2, r_psfFluxChi2, i_psfFluxChi2
FROM dp1.DiaObject
WHERE nDiaSources >= {N_SOURCES_MIN}
  AND (
       (g_psfFluxSigma > {FLUX_SIGMA_MIN} AND g_psfFluxStetsonJ > {STETSON_MIN})
    OR (r_psfFluxSigma > {FLUX_SIGMA_MIN} AND r_psfFluxStetsonJ > {STETSON_MIN})
    OR (i_psfFluxSigma > {FLUX_SIGMA_MIN} AND i_psfFluxStetsonJ > {STETSON_MIN})
  )
ORDER BY r_psfFluxSigma DESC
"""

print("Cerco oggetti variabili...")
job = service.submit_job(query_objects)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase:', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

dia_objects = job.fetch_result().to_table().to_pandas()
print(f"Trovati {len(dia_objects)} oggetti variabili candidati")
print(dia_objects[['diaObjectId', 'ra', 'dec', 'nDiaSources',
                    'r_psfFluxSigma', 'r_psfFluxStetsonJ']].to_string())

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — Scarica e salva curve di luce
# ─────────────────────────────────────────────
output_dir = "./light_curves"
os.makedirs(output_dir, exist_ok=True)

failed = []

for i, row in dia_objects.iterrows():
    obj_id = row['diaObjectId']
    print(f"\n[{i+1}/{len(dia_objects)}] diaObjectId={obj_id}")

    query_lc = f"""
    SELECT
        fsodo.diaObjectId,
        fsodo.coord_ra,
        fsodo.coord_dec,
        fsodo.visit,
        fsodo.band,
        fsodo.psfFlux,
        fsodo.psfFluxErr,
        fsodo.psfDiffFlux,
        fsodo.psfDiffFluxErr,
        vis.expMidptMJD
    FROM dp1.ForcedSourceOnDiaObject AS fsodo
    JOIN dp1.Visit AS vis ON vis.visit = fsodo.visit
    WHERE fsodo.diaObjectId = {obj_id}
    ORDER BY vis.expMidptMJD
    """

    try:
        job_lc = service.submit_job(query_lc)
        job_lc.run()
        job_lc.wait(phases=['COMPLETED', 'ERROR'])
        if job_lc.phase == 'ERROR':
            job_lc.raise_if_error()

        lc = job_lc.fetch_result().to_table().to_pandas()

        if len(lc) == 0:
            print(f"  Nessun dato, salto.")
            continue

        filename = os.path.join(output_dir, f"lc_{obj_id}.csv")
        lc.to_csv(filename, index=False)
        print(f"  Salvato: {filename}  ({len(lc)} punti, bande: {lc['band'].unique()})")

    except Exception as e:
        print(f"  ERRORE: {e}")
        failed.append(obj_id)

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — Salva catalogo oggetti
# ─────────────────────────────────────────────
catalog_path = os.path.join(output_dir, "variable_objects_catalog.csv")
dia_objects.to_csv(catalog_path, index=False)
print(f"\nCatalogo salvato: {catalog_path}")
if failed:
    print(f"Oggetti falliti ({len(failed)}): {failed}")
print("Done!")

In [3]:
import numpy as np
import pandas as pd
import os
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None

# ─────────────────────────────────────────────
# STEP 1a — Query ampia per crossmatch Gaia
# Nessun filtro su Sigma o StetsonJ
# ─────────────────────────────────────────────
query_all = """
SELECT
    diaObjectId,
    ra, dec,
    nDiaSources,
    r_psfFluxSigma,
    r_psfFluxStetsonJ,
    r_psfFluxChi2
FROM dp1.DiaObject
WHERE nDiaSources >= 10
ORDER BY nDiaSources DESC
"""

print("STEP 1a — Query ampia per crossmatch Gaia...")
job = service.submit_job(query_all)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase:', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

dia_objects_all = job.fetch_result().to_table().to_pandas()
print(f"Oggetti totali (per crossmatch Gaia): {len(dia_objects_all)}")

os.makedirs("./light_curves", exist_ok=True)
dia_objects_all.to_csv("./light_curves/catalog_for_gaia_xmatch.csv", index=False)
print("Salvato: catalog_for_gaia_xmatch.csv")

# ─────────────────────────────────────────────
# STEP 1b — Query ristretta per VAE
# Con StetsonJ e Sigma come prima
# (già disponibile, qui solo per riferimento)
# ─────────────────────────────────────────────
N_SOURCES_MIN  = 30
FLUX_SIGMA_MIN = 500.0
STETSON_MIN    = 0.5
MAX_OBJECTS    = 100000

query_vae = f"""
SELECT TOP {MAX_OBJECTS}
    diaObjectId,
    ra, dec,
    nDiaSources,
    g_psfFluxSigma, r_psfFluxSigma, i_psfFluxSigma,
    g_psfFluxMean,  r_psfFluxMean,  i_psfFluxMean,
    g_psfFluxStetsonJ, r_psfFluxStetsonJ, i_psfFluxStetsonJ,
    g_psfFluxChi2, r_psfFluxChi2, i_psfFluxChi2
FROM dp1.DiaObject
WHERE nDiaSources >= {N_SOURCES_MIN}
  AND (
       (g_psfFluxSigma > {FLUX_SIGMA_MIN} AND g_psfFluxStetsonJ > {STETSON_MIN})
    OR (r_psfFluxSigma > {FLUX_SIGMA_MIN} AND r_psfFluxStetsonJ > {STETSON_MIN})
    OR (i_psfFluxSigma > {FLUX_SIGMA_MIN} AND i_psfFluxStetsonJ > {STETSON_MIN})
  )
ORDER BY r_psfFluxSigma DESC
"""

print("\nSTEP 1b — Query ristretta per VAE...")
job2 = service.submit_job(query_vae)
job2.run()
job2.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase:', job2.phase)
if job2.phase == 'ERROR':
    job2.raise_if_error()

dia_objects_vae = job2.fetch_result().to_table().to_pandas()
print(f"Oggetti per VAE: {len(dia_objects_vae)}")

dia_objects_vae.to_csv("./light_curves/variable_objects_catalog.csv", index=False)
print("Salvato: variable_objects_catalog.csv")

# ─────────────────────────────────────────────
# STEP 2 — Scarica curve di luce (solo oggetti VAE)
# Download a batch per efficienza
# ─────────────────────────────────────────────
lc_dir = "./light_curves/lc"
os.makedirs(lc_dir, exist_ok=True)

# Resume automatico — salta già scaricati
already_done = set(
    int(f.replace("lc_", "").replace(".csv", ""))
    for f in os.listdir(lc_dir)
    if f.endswith(".csv")
)
print(f"\nGià scaricati: {len(already_done)}")

todo = dia_objects_vae[~dia_objects_vae['diaObjectId'].isin(already_done)]
print(f"Da scaricare: {len(todo)}")

BATCH_SIZE = 50
batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
print(f"Batch totali: {len(batches)}")

failed = []

for b_idx, batch in enumerate(batches):
    ids_str = ", ".join(str(i) for i in batch['diaObjectId'].tolist())

    print(f"[Batch {b_idx+1}/{len(batches)}] {len(batch)} oggetti...", end=" ")

    query_lc = f"""
    SELECT
        fsodo.diaObjectId,
        fsodo.coord_ra,
        fsodo.coord_dec,
        fsodo.visit,
        fsodo.band,
        fsodo.psfFlux,
        fsodo.psfFluxErr,
        fsodo.psfDiffFlux,
        fsodo.psfDiffFluxErr,
        vis.expMidptMJD
    FROM dp1.ForcedSourceOnDiaObject AS fsodo
    JOIN dp1.Visit AS vis ON vis.visit = fsodo.visit
    WHERE fsodo.diaObjectId IN ({ids_str})
    ORDER BY fsodo.diaObjectId, vis.expMidptMJD
    """

    try:
        job_lc = service.submit_job(query_lc)
        job_lc.run()
        job_lc.wait(phases=['COMPLETED', 'ERROR'])
        if job_lc.phase == 'ERROR':
            job_lc.raise_if_error()

        batch_lc = job_lc.fetch_result().to_table().to_pandas()

        if len(batch_lc) == 0:
            print("nessun dato.")
            continue

        for obj_id, lc in batch_lc.groupby('diaObjectId'):
            lc.to_csv(os.path.join(lc_dir, f"lc_{obj_id}.csv"), index=False)

        print(f"{batch_lc['diaObjectId'].nunique()} oggetti, {len(batch_lc)} punti.")

    except Exception as e:
        print(f"ERRORE: {e}")
        failed.extend(batch['diaObjectId'].tolist())

# ─────────────────────────────────────────────
# STEP 3 — Report finale
# ─────────────────────────────────────────────
downloaded = len([f for f in os.listdir(lc_dir) if f.endswith(".csv")])
print(f"\nDownload completato.")
print(f"  Curve scaricate:  {downloaded}")
print(f"  Batch falliti:    {len(failed)}")

if failed:
    pd.Series(failed, name='diaObjectId').to_csv(
        "./light_curves/failed_ids.csv", index=False)
    print("  IDs falliti salvati in: failed_ids.csv")

print("\nDone!")

STEP 1a — Query ampia per crossmatch Gaia...
Job phase: COMPLETED
Oggetti totali (per crossmatch Gaia): 51473
Salvato: catalog_for_gaia_xmatch.csv

STEP 1b — Query ristretta per VAE...
Job phase: COMPLETED
Oggetti per VAE: 13505
Salvato: variable_objects_catalog.csv

Già scaricati: 0
Da scaricare: 13505
Batch totali: 271
[Batch 1/271] 50 oggetti... 50 oggetti, 15899 punti.
[Batch 2/271] 50 oggetti... 50 oggetti, 11197 punti.
[Batch 3/271] 50 oggetti... 50 oggetti, 11009 punti.
[Batch 4/271] 50 oggetti... 50 oggetti, 8525 punti.
[Batch 5/271] 50 oggetti... 50 oggetti, 9073 punti.
[Batch 6/271] 50 oggetti... 50 oggetti, 7931 punti.
[Batch 7/271] 50 oggetti... 50 oggetti, 7087 punti.
[Batch 8/271] 50 oggetti... 50 oggetti, 8979 punti.
[Batch 9/271] 50 oggetti... 50 oggetti, 7087 punti.
[Batch 10/271] 50 oggetti... 50 oggetti, 7300 punti.
[Batch 11/271] 50 oggetti... 50 oggetti, 6979 punti.
[Batch 12/271] 50 oggetti... 50 oggetti, 9194 punti.
[Batch 13/271] 50 oggetti... 50 oggetti, 9046 p

In [3]:
import pandas as pd
import os
from lsst.rsp import get_tap_service

service = get_tap_service("tap")
assert service is not None

# ─────────────────────────────────────────────
# Carica il catalogo ampio
# ─────────────────────────────────────────────
catalog = pd.read_csv("./light_curves/catalog_for_gaia_xmatch.csv")
print(f"Oggetti totali nel catalogo: {len(catalog)}")

# ─────────────────────────────────────────────
# Cartella destinazione separata
# ─────────────────────────────────────────────
lc_dir = "./light_curves/lc2"
os.makedirs(lc_dir, exist_ok=True)

# Resume automatico — salta già scaricati in lc2/
already_done = set(
    int(f.replace("lc_", "").replace(".csv", ""))
    for f in os.listdir(lc_dir)
    if f.startswith("lc_") and f.endswith(".csv")
)
print(f"Già scaricate in lc2/: {len(already_done)}")

todo = catalog[~catalog['diaObjectId'].isin(already_done)]
print(f"Da scaricare:           {len(todo)}")

if len(todo) == 0:
    print("Niente da scaricare, tutto già presente.")
else:
    BATCH_SIZE = 50
    batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    print(f"Batch totali:           {len(batches)} (x{BATCH_SIZE} oggetti)\n")

    failed = []

    for b_idx, batch in enumerate(batches):
        ids_str = ", ".join(str(i) for i in batch['diaObjectId'].tolist())
        print(f"[Batch {b_idx+1}/{len(batches)}]", end=" ")

        query_lc = f"""
        SELECT
            fsodo.diaObjectId,
            fsodo.coord_ra,
            fsodo.coord_dec,
            fsodo.visit,
            fsodo.band,
            fsodo.psfFlux,
            fsodo.psfFluxErr,
            fsodo.psfDiffFlux,
            fsodo.psfDiffFluxErr,
            vis.expMidptMJD
        FROM dp1.ForcedSourceOnDiaObject AS fsodo
        JOIN dp1.Visit AS vis ON vis.visit = fsodo.visit
        WHERE fsodo.diaObjectId IN ({ids_str})
        ORDER BY fsodo.diaObjectId, vis.expMidptMJD
        """

        try:
            job_lc = service.submit_job(query_lc)
            job_lc.run()
            job_lc.wait(phases=['COMPLETED', 'ERROR'])
            if job_lc.phase == 'ERROR':
                job_lc.raise_if_error()

            batch_lc = job_lc.fetch_result().to_table().to_pandas()

            if len(batch_lc) == 0:
                print("nessun dato.")
                continue

            for obj_id, lc in batch_lc.groupby('diaObjectId'):
                lc.to_csv(os.path.join(lc_dir, f"lc_{obj_id}.csv"), index=False)

            print(f"{batch_lc['diaObjectId'].nunique()} oggetti, {len(batch_lc)} punti.")

        except Exception as e:
            print(f"ERRORE: {e}")
            failed.extend(batch['diaObjectId'].tolist())

    # ─────────────────────────────────────────────
    # Report finale
    # ─────────────────────────────────────────────
    total = len([f for f in os.listdir(lc_dir) if f.endswith(".csv")])
    print(f"\nDone!")
    print(f"  Totale curve in lc2/:  {total}")
    print(f"  Nuove scaricate:       {total - len(already_done)}")
    print(f"  Falliti:               {len(failed)}")

    if failed:
        pd.Series(failed, name='diaObjectId').to_csv(
            "./light_curves/failed_ids_lc2.csv", index=False)
        print("  IDs falliti salvati in: failed_ids_lc2.csv")

Oggetti totali nel catalogo: 51473
Già scaricate in lc2/: 0
Da scaricare:           51473
Batch totali:           1030 (x50 oggetti)

[Batch 1/1030] 50 oggetti, 37885 punti.
[Batch 2/1030] 50 oggetti, 33873 punti.
[Batch 3/1030] 50 oggetti, 32884 punti.
[Batch 4/1030] 50 oggetti, 30596 punti.
[Batch 5/1030] 50 oggetti, 26549 punti.
[Batch 6/1030] 50 oggetti, 26271 punti.
[Batch 7/1030] 50 oggetti, 24099 punti.
[Batch 8/1030] 50 oggetti, 23272 punti.
[Batch 9/1030] 50 oggetti, 19328 punti.
[Batch 10/1030] 50 oggetti, 17554 punti.
[Batch 11/1030] 50 oggetti, 15737 punti.
[Batch 12/1030] 50 oggetti, 16088 punti.
[Batch 13/1030] 50 oggetti, 14161 punti.
[Batch 14/1030] 50 oggetti, 15053 punti.
[Batch 15/1030] 50 oggetti, 16070 punti.
[Batch 16/1030] 50 oggetti, 13869 punti.
[Batch 17/1030] 50 oggetti, 13854 punti.
[Batch 18/1030] 50 oggetti, 12428 punti.
[Batch 19/1030] 50 oggetti, 14213 punti.
[Batch 20/1030] 50 oggetti, 14198 punti.
[Batch 21/1030] 50 oggetti, 13029 punti.
[Batch 22/1030

In [5]:
import os
import zipfile
from pathlib import Path

def zip_lc_folder(src_dir, zip_path):
    files = [f for f in os.listdir(src_dir) if f.endswith(".csv")]
    print(f"Comprimo {len(files)} file da {src_dir} → {zip_path}")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(files):
            zf.write(os.path.join(src_dir, fname), arcname=fname)
            if i % 1000 == 0:
                print(f"  {i}/{len(files)}...")
    
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"  Done! {size_mb:.1f} MB\n")

zip_lc_folder("./light_curves/lc",  "./light_curves/lc.zip")
zip_lc_folder("./light_curves/lc2", "./light_curves/lc2.zip")
print("Tutto compresso!")

Comprimo 0 file da ./light_curves/lc → ./light_curves/lc.zip
  Done! 0.0 MB

Comprimo 51473 file da ./light_curves/lc2 → ./light_curves/lc2.zip
  0/51473...
  1000/51473...
  2000/51473...
  3000/51473...
  4000/51473...
  5000/51473...
  6000/51473...
  7000/51473...
  8000/51473...
  9000/51473...
  10000/51473...
  11000/51473...
  12000/51473...
  13000/51473...
  14000/51473...
  15000/51473...
  16000/51473...
  17000/51473...
  18000/51473...
  19000/51473...
  20000/51473...
  21000/51473...
  22000/51473...
  23000/51473...
  24000/51473...
  25000/51473...
  26000/51473...
  27000/51473...
  28000/51473...
  29000/51473...
  30000/51473...
  31000/51473...
  32000/51473...
  33000/51473...
  34000/51473...
  35000/51473...
  36000/51473...
  37000/51473...
  38000/51473...
  39000/51473...
  40000/51473...
  41000/51473...
  42000/51473...
  43000/51473...
  44000/51473...
  45000/51473...
  46000/51473...
  47000/51473...
  48000/51473...
  49000/51473...
  50000/51473...
  